# Smart Healthcare Risk Prediction System
## Using Data Science and Neural Networks

This notebook performs end-to-end data analysis, visualization, statistical testing, and neural network classification on the **Medical Cost Personal Dataset** to predict patient health risk levels.

---

## Step 1 — Data Collection & Loading
Load the dataset using Python's built-in `open()` and then with **Pandas**.

In [ ]:
# 1.1 — Reading file using built-in open() function
with open('../dataset/insurance.csv', 'r') as f:
    header = f.readline()
    first_5_lines = [f.readline() for _ in range(5)]

print("Header:", header)
print("\nFirst 5 rows:")
for line in first_5_lines:
    print(line.strip())

In [ ]:
# 1.2 — Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("All libraries imported successfully!")

In [ ]:
# 1.3 — Load dataset with Pandas
df = pd.read_csv('../dataset/insurance.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {df.shape[0]}, Total Features: {df.shape[1]}")
df.head(10)

## Step 2 — Dataset Understanding & Exploration

In [ ]:
# 2.1 — Dataset Info
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
df.info()
print("\n")
print("Column Names:", list(df.columns))
print("Data Types:\n", df.dtypes)

In [ ]:
# 2.2 — Check for missing values
print("=" * 60)
print("MISSING VALUES CHECK")
print("=" * 60)
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# 2.3 — Handle missing values (demonstration even if no missing values)
# Fill numeric columns with median, categorical with mode
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handling:", df.isnull().sum().sum())

In [ ]:
# 2.4 — Unique values in categorical columns
print("=" * 60)
print("UNIQUE VALUES IN CATEGORICAL COLUMNS")
print("=" * 60)
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}: {df[col].unique()} (count: {df[col].nunique()})")

## Step 3 — Descriptive Statistics

In [ ]:
# 3.1 — Descriptive Statistics for numerical columns
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
df.describe().round(2)

In [ ]:
# 3.2 — Additional statistics
print("Skewness:\n", df.select_dtypes(include=[np.number]).skew().round(3))
print("\nKurtosis:\n", df.select_dtypes(include=[np.number]).kurtosis().round(3))
print("\nMedian values:\n", df.select_dtypes(include=[np.number]).median().round(2))

In [ ]:
# 3.3 — Value counts for categorical columns
print("=" * 60)
print("CATEGORICAL COLUMN VALUE COUNTS")
print("=" * 60)
for col in ['sex', 'smoker', 'region']:
    print(f"\n--- {col.upper()} ---")
    print(df[col].value_counts())
    print(f"Percentage:\n{(df[col].value_counts(normalize=True) * 100).round(2)}%")

## Step 4 — Feature Engineering
Create a **risk** column based on medical charges.

In [ ]:
# 4.1 — Create Risk Category (Target Variable)
df['risk'] = (df['charges'] > 15000).astype(int)

print("Risk Distribution:")
print(df['risk'].value_counts())
print(f"\nHigh Risk (1): {df['risk'].sum()} patients ({(df['risk'].mean()*100):.1f}%)")
print(f"Low Risk  (0): {(df['risk']==0).sum()} patients ({((df['risk']==0).mean()*100):.1f}%)")

In [ ]:
# 4.2 — Create BMI Category (Binning)
bins = [0, 18.5, 25, 30, 100]
labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
df['bmi_category'] = pd.cut(df['bmi'], bins=bins, labels=labels)

print("BMI Category Distribution:")
print(df['bmi_category'].value_counts())

In [ ]:
# 4.3 — Create Age Group (Binning)
age_bins = [0, 25, 35, 45, 55, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

print("Age Group Distribution:")
print(df['age_group'].value_counts().sort_index())

## Step 5 — Data Encoding & Preprocessing

In [ ]:
# 5.1 — Label Encoding for binary categorical variables
df['sex_encoded'] = df['sex'].map({'male': 1, 'female': 0})
df['smoker_encoded'] = df['smoker'].map({'yes': 1, 'no': 0})

print("Encoded columns created:")
print(df[['sex', 'sex_encoded', 'smoker', 'smoker_encoded']].head())

In [ ]:
# 5.2 — One-Hot Encoding for region (Indicator Variables)
region_dummies = pd.get_dummies(df['region'], prefix='region', dtype=int)
df = pd.concat([df, region_dummies], axis=1)

print("One-Hot Encoded Region columns:")
print(df[['region', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']].head())

In [ ]:
# 5.3 — Data Normalization using MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cols_to_normalize = ['age', 'bmi', 'children', 'charges']

df_normalized = df.copy()
df_normalized[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

print("Before Normalization:")
print(df[cols_to_normalize].describe().round(2))
print("\nAfter Normalization:")
print(df_normalized[cols_to_normalize].describe().round(2))

## Step 6 — Correlation Analysis

In [ ]:
# 6.1 — Correlation Matrix
numeric_cols = ['age', 'bmi', 'children', 'charges', 'sex_encoded', 'smoker_encoded', 'risk']
corr_matrix = df[numeric_cols].corr().round(3)

print("=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)
print(corr_matrix)

In [ ]:
# 6.2 — Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Heatmap - Healthcare Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../model/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey Insight: Smoking has the highest correlation with charges and risk.")

## Step 7 — ANOVA Test
Test whether features have statistically significant effects on medical charges.

In [ ]:
# 7.1 — ANOVA: Smoker vs Charges
smoker_yes = df[df['smoker'] == 'yes']['charges']
smoker_no = df[df['smoker'] == 'no']['charges']

f_stat, p_value = stats.f_oneway(smoker_yes, smoker_no)
print("=" * 60)
print("ANOVA TEST: Smoker vs Charges")
print("=" * 60)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.2e}")
print(f"Result: {'Significant (reject H0)' if p_value < 0.05 else 'Not significant (fail to reject H0)'}")
print(f"Conclusion: Smoking {'significantly' if p_value < 0.05 else 'does not significantly'} affects medical charges.")

In [ ]:
# 7.2 — ANOVA: Region vs Charges
regions = [df[df['region'] == r]['charges'] for r in df['region'].unique()]
f_stat, p_value = stats.f_oneway(*regions)

print("=" * 60)
print("ANOVA TEST: Region vs Charges")
print("=" * 60)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.4f}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'}")
print(f"Conclusion: Region {'significantly' if p_value < 0.05 else 'does not significantly'} affect medical charges.")

In [ ]:
# 7.3 — ANOVA: BMI Category vs Charges
bmi_groups = [df[df['bmi_category'] == cat]['charges'] for cat in df['bmi_category'].unique()]
f_stat, p_value = stats.f_oneway(*bmi_groups)

print("=" * 60)
print("ANOVA TEST: BMI Category vs Charges") 
print("=" * 60)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.2e}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'}")
print(f"Conclusion: BMI category {'significantly' if p_value < 0.05 else 'does not significantly'} affects medical charges.")

In [ ]:
# 7.4 — ANOVA: Sex vs Charges
male_charges = df[df['sex'] == 'male']['charges']
female_charges = df[df['sex'] == 'female']['charges']

f_stat, p_value = stats.f_oneway(male_charges, female_charges)

print("=" * 60)
print("ANOVA TEST: Sex vs Charges")
print("=" * 60)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.4f}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Not significant'}")

## Step 8 — Data Visualization (EDA)

In [ ]:
# 8.1 — Histogram: Distribution of BMI
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['bmi'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of BMI', fontsize=13, fontweight='bold')
axes[0].set_xlabel('BMI')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['bmi'].mean(), color='red', linestyle='--', label=f"Mean: {df['bmi'].mean():.1f}")
axes[0].legend()

axes[1].hist(df['age'], bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Age', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Frequency')

axes[2].hist(df['charges'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[2].set_title('Distribution of Medical Charges', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Charges ($)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../model/histograms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.2 — Scatter Plot: BMI vs Charges (colored by smoker)
plt.figure(figsize=(10, 6))
colors = {'yes': 'red', 'no': 'green'}
for smoker_status in ['yes', 'no']:
    subset = df[df['smoker'] == smoker_status]
    plt.scatter(subset['bmi'], subset['charges'], 
                c=colors[smoker_status], alpha=0.5, 
                label=f'Smoker: {smoker_status}', edgecolors='black', linewidth=0.3)

plt.title('BMI vs Medical Charges (by Smoking Status)', fontsize=14, fontweight='bold')
plt.xlabel('BMI', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../model/scatter_bmi_charges.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.3 — Pie Chart: Smokers vs Non-Smokers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

smoker_counts = df['smoker'].value_counts()
axes[0].pie(smoker_counts, labels=['Non-Smoker', 'Smoker'], 
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.05), shadow=True,
            textprops={'fontsize': 12})
axes[0].set_title('Smoker vs Non-Smoker Distribution', fontsize=13, fontweight='bold')

risk_counts = df['risk'].value_counts()
axes[1].pie(risk_counts, labels=['Low Risk', 'High Risk'],
            autopct='%1.1f%%', colors=['#3498db', '#e74c3c'],
            startangle=90, explode=(0, 0.05), shadow=True,
            textprops={'fontsize': 12})
axes[1].set_title('Risk Level Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../model/pie_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.4 — Bar Chart: Region-wise Average Medical Cost
plt.figure(figsize=(10, 6))
region_avg = df.groupby('region')['charges'].mean().sort_values(ascending=False)
bars = plt.bar(region_avg.index, region_avg.values, 
               color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'],
               edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, region_avg.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'${val:,.0f}', ha='center', fontsize=11, fontweight='bold')

plt.title('Region-wise Average Medical Cost', fontsize=14, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Average Charges ($)', fontsize=12)
plt.tight_layout()
plt.savefig('../model/bar_region_charges.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.5 — Box Plot: Smoking vs Charges
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(x='smoker', y='charges', data=df, ax=axes[0], 
            palette={'yes': '#e74c3c', 'no': '#2ecc71'})
axes[0].set_title('Smoking Status vs Medical Charges', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Smoker', fontsize=12)
axes[0].set_ylabel('Charges ($)', fontsize=12)

sns.boxplot(x='risk', y='bmi', data=df, ax=axes[1],
            palette={0: '#3498db', 1: '#e74c3c'})
axes[1].set_title('Risk Level vs BMI', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Risk (0=Low, 1=High)', fontsize=12)
axes[1].set_ylabel('BMI', fontsize=12)

plt.tight_layout()
plt.savefig('../model/box_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.6 — Scatter Plot: Age vs Charges (by Risk Level)
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['age'], df['charges'], c=df['risk'], 
                       cmap='RdYlGn_r', alpha=0.6, edgecolors='black', linewidth=0.3)
plt.colorbar(scatter, label='Risk Level (0=Low, 1=High)')
plt.axhline(y=15000, color='red', linestyle='--', linewidth=2, label='Risk Threshold ($15,000)')
plt.title('Age vs Medical Charges (by Risk Level)', fontsize=14, fontweight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../model/scatter_age_charges.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8.7 — Pairplot for key features
sns.pairplot(df[['age', 'bmi', 'charges', 'smoker']], hue='smoker',
             palette={'yes': 'red', 'no': 'green'}, diag_kind='hist',
             plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of Key Features', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('../model/pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Linear Regression (Charges Prediction)

In [ ]:
# 9.1 — Linear Regression for Charges Prediction
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Prepare features for regression
feature_cols = ['age', 'bmi', 'children', 'sex_encoded', 'smoker_encoded',
                'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']

X_reg = df[feature_cols]
y_reg = df['charges']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train_r, y_train_r)
y_pred_r = lr_model.predict(X_test_r)

print("=" * 60)
print("LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R² Score: {r2_score(y_test_r, y_pred_r):.4f}")
print(f"Mean Absolute Error: ${mean_absolute_error(y_test_r, y_pred_r):,.2f}")
print(f"Root Mean Squared Error: ${np.sqrt(mean_squared_error(y_test_r, y_pred_r)):,.2f}")

print("\nFeature Coefficients:")
coeff_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lr_model.coef_}).sort_values('Coefficient', ascending=False)
print(coeff_df.to_string(index=False))

In [ ]:
# 9.2 — Regression: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test_r, y_pred_r, alpha=0.5, color='steelblue', edgecolors='black', linewidth=0.3)
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.title('Linear Regression: Actual vs Predicted Charges', fontsize=14, fontweight='bold')
plt.xlabel('Actual Charges ($)', fontsize=12)
plt.ylabel('Predicted Charges ($)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../model/regression_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 — Neural Network Model (MLP Classifier)
Train a Multi-Layer Perceptron for risk classification.

In [ ]:
# 10.1 — Prepare Data for Neural Network
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# Features and target
X = df[feature_cols].values
y = df['risk'].values

# Scale features for neural network
scaler_nn = StandardScaler()
X_scaled = scaler_nn.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")

In [ ]:
# 10.2 — Train MLP Neural Network
# Architecture: Input(9) → Hidden(8) → Hidden(4) → Output(1)
mlp = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42,
    learning_rate='adaptive',
    early_stopping=True,
    validation_fraction=0.1
)

mlp.fit(X_train, y_train)

print("=" * 60)
print("NEURAL NETWORK TRAINING COMPLETE")
print("=" * 60)
print(f"Architecture: Input({X_train.shape[1]}) → Hidden(8) → Hidden(4) → Output(1)")
print(f"Activation: ReLU (hidden), Sigmoid (output)")
print(f"Solver: Adam")
print(f"Iterations completed: {mlp.n_iter_}")
print(f"Final Loss: {mlp.loss_:.4f}")

In [ ]:
# 10.3 — Model Evaluation
y_pred = mlp.predict(X_test)

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
# 10.4 — Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])
plt.title('Confusion Matrix - MLP Neural Network', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('../model/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTrue Positives: {cm[1][1]}")
print(f"True Negatives: {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")

In [ ]:
# 10.5 — Training Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(mlp.loss_curve_, color='steelblue', linewidth=2)
plt.title('Neural Network Training Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Iterations', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../model/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10.6 — Feature Importance (using permutation importance)
from sklearn.inspection import permutation_importance

result = permutation_importance(mlp, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': result.importances_mean
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue', edgecolor='black')
plt.title('Feature Importance - MLP Neural Network', fontsize=14, fontweight='bold')
plt.xlabel('Mean Importance', fontsize=12)
plt.tight_layout()
plt.savefig('../model/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10.7 — Decision Boundary Visualization (2D projection: BMI vs Age)
from matplotlib.colors import ListedColormap

# Train a simpler 2D model for visualization
X_2d = df[['bmi', 'age']].values
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

mlp_2d = MLPClassifier(hidden_layer_sizes=(8, 4), activation='relu', max_iter=1000, random_state=42)
mlp_2d.fit(X_2d_scaled, y)

# Create mesh grid
x_min, x_max = X_2d_scaled[:, 0].min() - 0.5, X_2d_scaled[:, 0].max() + 0.5
y_min, y_max = X_2d_scaled[:, 1].min() - 0.5, X_2d_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
Z = mlp_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#2ecc71', '#e74c3c']))
scatter = plt.scatter(X_2d_scaled[:, 0], X_2d_scaled[:, 1], c=y, 
                       cmap=ListedColormap(['green', 'red']), alpha=0.5, edgecolors='black', linewidth=0.3)
plt.title('Decision Boundary - MLP (BMI vs Age)', fontsize=14, fontweight='bold')
plt.xlabel('BMI (scaled)', fontsize=12)
plt.ylabel('Age (scaled)', fontsize=12)
plt.colorbar(scatter, label='Risk (0=Low, 1=High)')
plt.tight_layout()
plt.savefig('../model/decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11 — Save Model for Streamlit Deployment

In [ ]:
# 11.1 — Save the trained model and scaler
import joblib

# Save MLP classifier
joblib.dump(mlp, '../model/mlp_model.pkl')
print("MLP model saved to: model/mlp_model.pkl")

# Save the scaler
joblib.dump(scaler_nn, '../model/scaler.pkl')
print("Scaler saved to: model/scaler.pkl")

# Save Linear Regression model
joblib.dump(lr_model, '../model/lr_model.pkl')
print("Linear Regression model saved to: model/lr_model.pkl")

# Save feature columns for reference
joblib.dump(feature_cols, '../model/feature_cols.pkl')
print("Feature columns saved to: model/feature_cols.pkl")

print("\n✅ All models and artifacts saved successfully!")

## Step 12 — Geospatial Visualization (Bonus)

In [ ]:
# 12.1 — Geospatial Map using Folium
import folium

# Map US regions to approximate coordinates
region_coords = {
    'southeast': [33.7, -84.4],   # Atlanta, GA
    'southwest': [33.4, -112.0],  # Phoenix, AZ
    'northeast': [40.7, -74.0],   # New York, NY
    'northwest': [47.6, -122.3]   # Seattle, WA
}

region_stats = df.groupby('region').agg(
    avg_charges=('charges', 'mean'),
    avg_bmi=('bmi', 'mean'),
    total_patients=('charges', 'count'),
    high_risk_pct=('risk', 'mean')
).round(2)

# Create map centered on US
m = folium.Map(location=[39.8, -98.5], zoom_start=4, tiles='OpenStreetMap')

for region, row in region_stats.iterrows():
    coords = region_coords[region]
    color = 'red' if row['high_risk_pct'] > 0.3 else 'orange' if row['high_risk_pct'] > 0.25 else 'green'
    
    popup_html = f"""
    <b>Region: {region.upper()}</b><br>
    Avg Charges: ${row['avg_charges']:,.2f}<br>
    Avg BMI: {row['avg_bmi']:.1f}<br>
    Total Patients: {int(row['total_patients'])}<br>
    High Risk: {row['high_risk_pct']*100:.1f}%
    """
    
    folium.CircleMarker(
        location=coords,
        radius=row['total_patients'] / 10,
        popup=folium.Popup(popup_html, max_width=200),
        color=color,
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{region}: ${row['avg_charges']:,.0f}"
    ).add_to(m)

# Save map
m.save('../model/region_map.html')
print("Geospatial map saved to: model/region_map.html")
m

## Summary

### Key Findings:
1. **Smoking** is the strongest predictor of high medical costs and risk
2. **BMI** and **Age** also significantly contribute to risk prediction
3. **ANOVA** confirms smoking status has a statistically significant effect on charges
4. The **MLP Neural Network** achieves high accuracy in classifying patients as High/Low Risk
5. **Linear Regression** provides reasonable charge predictions

### Models Saved:
- `mlp_model.pkl` — Neural Network classifier
- `lr_model.pkl` — Linear Regression for charge prediction
- `scaler.pkl` — Feature scaler

### Next Step:
Run the **Streamlit App** (`frontend/app.py`) for interactive predictions.